# Lab 4 · Fine-tuning (ConvNeXt-Tiny, EfficientNetV2-S, Swin-T) · búsqueda de hiperparámetros · Kaggle

Mismas celdas de preparación que los notebooks de Kaggle anteriores (modo determinista estricto, semilla 42, recorte
224, aumentos en GPU, normalización ImageNet). Tres redes preentrenadas en ImageNet de torchvision con una capa final
nueva de 7 salidas; si una no pasa la prueba de determinismo se sustituye por ResNet-50.

- **Fase 1:** 10 configuraciones aleatorias por red (la 00 es una receta estándar), 10 épocas, poda por mediana en la
  época 4. Se busca: lr del backbone, multiplicador de lr de la cabeza (1 o 10), weight decay, batch, warm-up,
  label smoothing, pesos de clase, mixup, aumentos y stochastic depth.
- **Fase 2:** las 2 mejores de cada red, 30 épocas, se guarda el `.pt` de la mejor época de validación.
- **Final:** la mejor de la fase 2 → `final_ft/output_ft.csv` (+ `codabench_submission.zip` si `output_custom.csv`
  está entre los inputs).

1. *Add Input* → el dataset con `db.zip` (y, opcional, un dataset con `output_custom.csv`). *Accelerator* → **GPU T4**.
   *Settings* → **Internet ON** (descarga los pesos de ImageNet).
2. **Save Version → Save & Run All**. Como mucho 11 h (el planificador reserva tiempo para la fase 2).
3. *Output*: `outputs/Lab_4_2026_2027/ft/` (`ft_results.csv`, `ft_leaderboard_phase1/2.csv`, `ft_per_class_f1.csv`,
   `*_epochs.csv`, `*_p2_*.pt`, `determinism_check.json`) y `final_ft/`.

In [1]:
%matplotlib inline

In [2]:
import os
from pathlib import Path
import copy
import numbers
import random
import time
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image
from skimage import io, transform, util

import torch  
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils, models


# Install missing dependencies in your chosen environment before running.
# In a notebook, you can use:
# %pip install torch torchvision pandas scikit-image matplotlib scikit-learn pillow

In [3]:
# ---- Determinism (must run before any CUDA work) ----
# Strict mode: any operation without a deterministic implementation raises an error instead of
# silently giving run-to-run differences. Together with fixed seeds this makes every run repeatable.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'
import torch
import torch._utils            # load PyTorch internals up front: AdamW imports torch._dynamo lazily,
import torch._dynamo           # and in the Kaggle torch build that lazy import fails if done later
torch.use_deterministic_algorithms(True)
print('torch', torch.__version__, '| deterministic:', torch.are_deterministic_algorithms_enabled())

torch 2.11.0+cu130 | deterministic: True


In [4]:
SEED = 42
CLASS_NAMES = ['Water', 'Very Low', 'Low', 'Moderate', 'High', 'Very High', 'Non-burnable']
NUM_CLASSES = len(CLASS_NAMES)
MAX_TRAIN_SAMPLES = 0  # Set to 0 to use the full training split. 500 is a good number for quick experiments
# Windows cannot use worker processes with notebook-defined classes -> 0. Linux: up to 4 workers.
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 1)
# Epoch: num. of iterations over the entire dataset over batches.
# Between those iterations the model weights are updated based on the loss function and optimizer.
CUSTOM_EPOCHS = 30
FT_EPOCHS = 5
# LERNING RATE: Step Size of the Optimizer at each iteration while moving toward a minimum of a loss function. Affects how weights are updated
CUSTOM_LR = 1e-2
FT_LR = 1e-3

# Note: PIXEL_MEAN and PIXEL_STD come directly from AlexNet's
# documentation in PyTorch:
# https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html
# They are the per-channel (R, G, B) mean and standard deviation of the ImageNet training set,
# the dataset the pretrained torchvision models (AlexNet, etc.) were trained on. Normalising
# with them centres each channel around 0 with unit variance (stable, faster training) and,
# for the fine-tuned network, gives it the input statistics it was pretrained with.
# For a network trained from scratch they are not mandatory (the dataset's own mean/std
# could be used instead), but they are a reasonable default.
PIXEL_MEAN = [0.485, 0.456, 0.406]
PIXEL_STD = [0.229, 0.224, 0.225]

DATA_ARCHIVE = Path('db.zip')
DATA_ROOT = Path('data')
DB_DIR = DATA_ROOT / 'db'
OUTPUT_DIR = Path('outputs') / 'Lab_4_2026_2027'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Class order: {CLASS_NAMES}')

Device: cuda:0
Class order: ['Water', 'Very Low', 'Low', 'Moderate', 'High', 'Very High', 'Non-burnable']


In [5]:
# ---- Platform setup: Kaggle (dataset from "Add Input") or Google Colab (db.zip in Google Drive) ----
# The SAME notebook runs on both; each platform trains a different seed (Kaggle 42, Colab 43), so every
# "CutMix vs reference" comparison happens inside one machine.
import shutil, sys, time
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = (not IN_COLAB) and Path('/kaggle/input').exists()   # Colab also has /kaggle/input now
COLAB_DB_ZIP = '/content/drive/MyDrive/L4/db.zip'                 # <-- Colab: where db.zip is in your Drive
COLAB_WORK = Path('/content/drive/MyDrive/L4/cutmix_colab')      # <-- Colab: results go to Drive (survive disconnects)
WORK = Path('/kaggle/working') if IN_KAGGLE else (COLAB_WORK if IN_COLAB else Path('.'))
WORK.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ARCHIVE = Path('/content/db.zip')                        # local copy: extracting from Drive can disconnect
    if not DATA_ARCHIVE.exists() or DATA_ARCHIVE.stat().st_size != Path(COLAB_DB_ZIP).stat().st_size:
        shutil.copy(COLAB_DB_ZIP, DATA_ARCHIVE)
    DATA_ROOT = Path('/content/data')
    DB_DIR = DATA_ROOT / 'db'
if IN_KAGGLE:
    zips = list(Path('/kaggle/input').rglob('db.zip'))
    csvs = list(Path('/kaggle/input').rglob('train_reduced.csv'))
    if zips:
        DATA_ARCHIVE = WORK / 'db.zip'
        if not DATA_ARCHIVE.exists():
            shutil.copy(zips[0], DATA_ARCHIVE)
        DATA_ROOT = WORK / 'data'
        DB_DIR = DATA_ROOT / 'db'
    elif csvs:
        DB_DIR = csvs[0].parent
        DATA_ARCHIVE = Path('/nonexistent.zip')
    else:
        raise FileNotFoundError('Add the FireRisk dataset with "Add Input" (db.zip or its extracted db/ folder).')
print('DB_DIR =', DB_DIR)

DB_DIR = /kaggle/input/datasets/csarlvarezcascos/database/db


In [6]:
required_csvs = [DB_DIR / f'{split}_reduced.csv'
                 for split in ('train', 'val', 'test')]

if DATA_ARCHIVE.is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    restored_files = 0
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:

        for member in archive.infolist():
            if not member.filename.startswith('db/'):
                continue
            target = DATA_ROOT / member.filename
            if not member.is_dir() and target.is_file() and target.stat().st_size == member.file_size:
                continue
            if not target.resolve().is_relative_to(DATA_ROOT.resolve()):
                raise ValueError('The archive contains an invalid dataset path.')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            elif not target.is_file() or target.stat().st_size != member.file_size:
                archive.extract(member, DATA_ROOT)
                restored_files += 1
    print(f'Dataset ready at {DB_DIR.resolve()}; restored {restored_files} files.')

if not all(path.is_file() for path in required_csvs):
    raise FileNotFoundError(
        'Dataset CSV files were not found. Start the kernel in the notebook directory '
        'or update DATA_ARCHIVE and DATA_ROOT in the setup cell.')

for csv_path in required_csvs:
    ids = pd.read_csv(csv_path, usecols=['id'], dtype={'id': str})['id']
    for image_id in ids:
        image_path = DB_DIR / 'images' / f'{image_id}.jpg'
        if not image_path.is_file() or image_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'Missing or empty image: {image_path}. Supply the original db.zip '
                'and rerun this setup cell to restore it.')

print('All indexed images exist and are nonempty.')

All indexed images exist and are nonempty.


In [7]:
class FireRiskDataset(Dataset):
    """Read RGB images and integer labels using a CSV index.

    Args:
        csv_file: CSV with columns 'id' and 'label'.
        root_dir: Dataset directory containing the 'images' subdirectory.
        transform: Optional callable that receives and returns a sample dictionary.
        maxSize: Maximum subset size; 0 selects the full split.
    """

    # Creates and checks the dataset
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0, preload=False):

        if not isinstance(maxSize, numbers.Integral) or maxSize < 0:
            raise ValueError('maxSize must be a nonnegative integer.')

        self.dataset = pd.read_csv(csv_file, dtype={'id': str, 'label': int})

        if not {'id', 'label'}.issubset(self.dataset.columns) or self.dataset.empty:
            raise ValueError('The CSV must contain nonempty id and label columns.')
        if not self.dataset['label'].isin(range(-1, NUM_CLASSES)).all():
            raise ValueError('Labels must be -1 (hidden test labels) or 0 through 6.')
        if maxSize > 0:
            indices = np.random.RandomState(SEED).permutation(len(self.dataset)) # shuffle the dataset
            self.dataset = self.dataset.iloc[indices[:maxSize]].reset_index(drop=True) # after shuffling, select 1st maxSize rows

        self.img_dir = Path(root_dir) / 'images'
        self.transform = transform
        self.classes = CLASS_NAMES.copy()

        # DECODE IMG in RAM to SPEED UP TRAINING. Optional (default preload is False and wouldn't be used if memory is limited)
        self.cache = None
        if preload:
            first = self._read(0)
             # Array of size (len(dataset), H, W, 3) to store all images in RAM:
            self.cache = np.empty((len(self.dataset), *first.shape), dtype=np.uint8)
            for i in range(len(self.dataset)):
                self.cache[i] = self._read(i) # Store each image in the cache

    def _read(self, idx):
        """Read one image FROM DISK as an H x W x 3 uint8 array"""
        image = io.imread(self.img_dir / f"{self.dataset.iloc[int(idx)]['id']}.jpg")
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2) # Convert grayscale to RGB
        elif image.ndim == 3 and image.shape[2] == 4:
            image = image[..., :3] # Discard alpha channel if present
        if image.ndim != 3 or image.shape[2] != 3: 
            raise ValueError('Expected an RGB image with three channels.')
        return image


    def __len__(self):
        return len(self.dataset)

    # Get a sample from the dataset. Each sample is a dictionary: {'image': image, 'label': label}.
    # When doing dataset[i], the __getitem__ method is called, where i is the index and ['image'] is the image
    def __getitem__(self, idx):

        if torch.is_tensor(idx): 
            idx = idx.item()
        idx = int(idx)
        raw = self.cache[idx] if self.cache is not None else self._read(idx) # Read from cache if available
        sample = {'image': util.img_as_float32(raw), 'label': int(self.dataset.iloc[idx]['label'])}

        # APPLY TRANSFORMATIONS (if any) to the sample before returning it
        return self.transform(sample) if self.transform is not None else sample

In [8]:
def _crop_size(output_size):
    # Returns a tuple representing the crop size

    if isinstance(output_size, numbers.Integral):
        output_size = (int(output_size), int(output_size)) # Cropped image is square if only one integer is provided

    if (not isinstance(output_size, tuple) or len(output_size) != 2 or
        any(not isinstance(v, numbers.Integral) or v <= 0 for v in output_size)):
        raise ValueError('Size must be a positive integer or a pair of positive integers.')

    return tuple(int(v) for v in output_size)

class Rescale:
    """Resize to (height, width), or resize the shorter side preserving aspect ratio."""
    # If output_size is an integer, resize calculating the scale of that integer with respect to the short side

    def __init__(self, output_size):
        self.output_size = (int(output_size) if isinstance(output_size, numbers.Integral)
                            else _crop_size(output_size))
        if isinstance(self.output_size, int) and self.output_size <= 0:
            raise ValueError('The output size must be positive.')

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            scale = self.output_size / min(h, w)
            new_h, new_w = round(h * scale), round(w * scale)
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (new_h, new_w), anti_aliasing=True)

        return {'image': image.astype(np.float32), 'label': label}

class RandomCrop:
    """Choose uniformly among all valid crops, including the final row/column."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        # Choose startring coordinates for the crop randomly within the valid range to avoid going out of bounds
        top = np.random.randint(0, h - new_h + 1)
        left = np.random.randint(0, w - new_w + 1)

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class CenterCrop:
    """Take a deterministic central crop without silently changing its size."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        top, left = (h - new_h) // 2, (w - new_w) // 2

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class ToTensor:
    """Convert HWC images to contiguous float32 CHW tensors and labels to int64."""

    def __call__(self, sample):
        image = np.ascontiguousarray(sample['image'].transpose(2, 0, 1), dtype=np.float32)
        return {'image': torch.from_numpy(image), # Numpy array converted to PyTorch tensor
                'label': torch.tensor(sample['label'], dtype=torch.long)} # Label converted to PyTorch tensor

class Normalize:
    """Apply per-channel normalisation without mutating the input tensor."""

    def __init__(self, mean, std):

        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)

        if (self.mean.ndim != 1 or self.std.shape != self.mean.shape or
            not np.isfinite(self.mean).all() or not np.isfinite(self.std).all() or
            not (self.std > 0).all()):
            raise ValueError('Mean/std must be matching finite vectors with positive std.')

    def __call__(self, sample):

        image = sample['image']

        if image.ndim != 3 or image.shape[0] != len(self.mean):
            raise ValueError('The CHW tensor and normalisation vectors must match.')

        mean = torch.as_tensor(self.mean, dtype=image.dtype, device=image.device)[:, None, None] # convert mean to tensor
        std = torch.as_tensor(self.std, dtype=image.dtype, device=image.device)[:, None, None]
        return {'image': (image - mean) / std, 'label': sample['label']}

In [9]:
class GPUAugment:
    """Random augmentation + normalisation of a batch (N x 3 x H x W, floats in [0, 1]) on the GPU."""
    def __init__(self, mean, std, gamma=0.1, brightness=0.05, contrast=0.05, saturation=0.05):
        self.mean = torch.tensor(mean).view(1, 3, 1, 1)
        self.std = torch.tensor(std).view(1, 3, 1, 1)
        self.gamma, self.brightness, self.contrast, self.saturation = gamma, brightness, contrast, saturation

    @staticmethod
    def _factor(n, amount, device):
        return 1 + (torch.rand(n, 1, 1, 1, device=device) * 2 - 1) * amount

    @staticmethod
    def _gray(x):
        return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]

    def __call__(self, x):
        n, dev = x.shape[0], x.device
        pick = lambda p: (torch.rand(n, device=dev) < p).view(n, 1, 1, 1)
        x = torch.where(pick(0.5), x.flip(3), x)                    # horizontal flip
        x = torch.where(pick(0.5), x.flip(2), x)                    # vertical flip
        k = torch.randint(0, 4, (n,), device=dev).view(n, 1, 1, 1)
        x = torch.where(k == 0, x, torch.where(k == 1, x.rot90(1, (2, 3)),
                        torch.where(k == 2, x.rot90(2, (2, 3)), x.rot90(3, (2, 3)))))
        x = x.clamp(1e-6, 1) ** self._factor(n, self.gamma, dev)    # gamma
        x = x * self._factor(n, self.brightness, dev)               # brightness
        m = self._gray(x).mean(dim=(1, 2, 3), keepdim=True)
        x = (x - m) * self._factor(n, self.contrast, dev) + m       # contrast
        g = self._gray(x)
        x = (x - g) * self._factor(n, self.saturation, dev) + g     # saturation
        return (x.clamp(0, 1) - self.mean.to(dev)) / self.std.to(dev)


class GPULoader:
    """Wraps a DataLoader: moves each batch to the device and applies `augment` there."""
    def __init__(self, loader, augment):
        self.loader, self.augment = loader, augment
        self.dataset, self.generator = loader.dataset, loader.generator   # train_model uses these

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        for sample in self.loader:
            x = sample['image'].to(device, dtype=torch.float32, non_blocking=True)
            yield {'image': self.augment(x), 'label': sample['label']}

In [10]:
train_dataset = FireRiskDataset(
    DB_DIR / 'train_reduced.csv', DB_DIR, maxSize=MAX_TRAIN_SAMPLES, preload=False,
    transform=transforms.Compose([
        RandomCrop(224), ToTensor()  
        ]))  # transform could be also = train_aug, but we want to use the GPUAugment class, which already normalizes and augments

val_dataset = FireRiskDataset(
    DB_DIR / 'val_reduced.csv', DB_DIR,
    transform=transforms.Compose([
        CenterCrop(224), ToTensor(), Normalize(PIXEL_MEAN, PIXEL_STD),
    ]))

test_dataset = FireRiskDataset(
    DB_DIR / 'test_reduced.csv', DB_DIR,
    transform=transforms.Compose([
        CenterCrop(224), ToTensor(), Normalize(PIXEL_MEAN, PIXEL_STD),
    ]))

for i in range(min(4, len(train_dataset))):
    sample = train_dataset[i]
    print(i, sample['image'].shape, int(sample['label']))

print(f'Training subset size: {len(train_dataset)}')

0 torch.Size([3, 224, 224]) 0
1 torch.Size([3, 224, 224]) 0
2 torch.Size([3, 224, 224]) 0
3 torch.Size([3, 224, 224]) 0
Training subset size: 3516


In [11]:
def seed_worker(worker_id):
    # PyTorch assigns a worker seed; also seed NumPy and Python random.
    worker_seed = torch.initial_seed() % (2 ** 32) # ensure its within the range of a 32-bit integer
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Torch Generator used to seed the workers for reproducibility. 
# Generator() is a random number generator that can be used to control the randomness of operations in PyTorch
train_generator = torch.Generator().manual_seed(SEED) # Fixes the shuffle order 

# pin_memory speeds up CPU -> GPU copies; persistent_workers avoids re-spawning workers every epoch.
    # In windows, persistent_workers is ignored, as NUM_WORKERS is 0.
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT = NUM_WORKERS > 0

# Batch size can be an hyperparameter in train (in val and test it doesnt change predictions)
train_dataloader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, generator=train_generator,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

# In val and test we dont compute gradients so we can use a larger batch size.
val_dataloader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

test_dataloader = DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

In [12]:
# ---- Architecture HYPERPARAMETERS ( to edit here, not inside the class) ----
INPUT_SIZE = 224                 # side of the square crop given to the network (see RandomCrop/CenterCrop)

# Initially, this is the reference setup. TODO: explore other architectures, hyperparams, methods and compare validations
CUSTOM_ARCH = dict(
    channels=(32, 64, 128, 256),        # nº of filters of each conv block (nº filters = nº blocks)
    kernel_size=3,               # conv kernel (stride 1)
    padding=1,                   # 0 shrinks the map; kernel_size // 2 keeps its size
    hidden=128,                  # out units of the first fully connected layer
    dropout=0.5,                 # dropout probability before the last layer (0 = off)
    batch_norm=True,            # BatchNorm after each conv
    convs_per_block=1,           # convs (Conv-BN-ReLU) before each MaxPool; 2 = VGG style
    head='flatten',              # 'flatten' = Flatten -> fc1 (reference); 'gap' = global average pooling -> fc1
)


class CustomNet(nn.Module):
    """CNN from scratch: N x 3 x S x S  ->  N x 7 raw logits (no softmax inside, softmax is implemented in loss function).

    [[Conv(k x k) -> (BatchNorm) -> ReLU] x convs_per_block -> MaxPool(2 x 2)] x len(channels) -> Flatten (or global
    average pooling if head='gap') -> fc1 -> ReLU -> (Dropout) -> fc2

    The input size of fc1 is autocomputed from input_size, so changing the input size, kernel,
    padding or number of blocks does not require editing the linear layer.
    """

    def __init__(self, channels=(6, 16, 32), kernel_size=5, padding=0, hidden=128,
                 dropout=0.0, batch_norm=False, input_size=INPUT_SIZE, num_classes=NUM_CLASSES,
                 convs_per_block=1, head='flatten'):
        super().__init__()
        if head not in ('flatten', 'gap'):
            raise ValueError("head must be 'flatten' or 'gap'.")

        blocks, in_channels, in_size = [], 3, input_size
        for out_channels in channels:
            for _ in range(convs_per_block):   # first conv: in_channels -> out_channels, the others: out_channels -> out_channels
                blocks.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding))

                # After each conv, before ReLu, to normalize the activations and gradients, improving training stability and speed.
                if batch_norm:
                    blocks.append(nn.BatchNorm2d(out_channels))
                blocks.append(nn.ReLU(inplace=True))

                in_size = in_size - kernel_size + 2 * padding + 1   # conv: size - k + 2p + 1
                # For the next conv, the input channels are the output channels of this one
                in_channels = out_channels
            # After the convs of the block, Pool
            blocks.append(nn.MaxPool2d(2))
            in_size //= 2                                           # pool: floor(size / 2)
        if in_size < 1:
            raise ValueError('The input is too small for this number of blocks.')

        # We call features to the sequence of Conv LAYERS created, it can be called in forward() to compute the feature maps of the input images.
        self.features = nn.Sequential(*blocks) 

        # FC1 Layer depends on input size and nº of blocks: compute number of features for the first FC layer
        # head='gap': global average pooling leaves one value per channel; 'flatten': the whole map (C*H*W)
        self.pool = nn.AdaptiveAvgPool2d(1) if head == 'gap' else nn.Identity()
        flat_features = in_channels if head == 'gap' else in_channels * in_size * in_size      # in_channels is the last one computed (last block out_channels)   

        # AFTER CONV BLOCKS:  Sequence of FC layers: Linear -> ReLU -> Dropout -> Linear (output)
            # It is the classifier part of the network, takes the flattened feature maps and produces the final class logits.            
        self.classifier = nn.Sequential(
            nn.Linear(flat_features, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(hidden, num_classes))

        # He initialisation for ReLU convs
        for m in self.modules():  # For every layer in the network
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu') 
                nn.init.zeros_(m.bias)

    # Forward pass: x is a batch of images (N x 3 x Size x Size), returns N x 7 logits
    def forward(self, x):
        x = self.pool(self.features(x))   # N x C x H x W  (N x C x 1 x 1 with head='gap')
        # self.classifier receives a flattened input as its 1st layer is Linear
        x = x.flatten(1)             # N x (C*H*W) 
        return self.classifier(x)    # N x 7 logits

In [13]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

In [14]:
import math
def train_model(model, criterion, optimizer, scheduler,
                num_epochs=25, plot_confusion_matrix=True, *,
                loaders=None, amp=False, mixup_alpha=0.0, epoch_callback=None, verbose=True,
                cutmix_alpha=0.0, cutmix_mode='all', mix_prob=1.0):
    """Run fixed-budget training and restore the best validation-macro-F1 model.

    Defaults reproduce the plain loop. Optional: `loaders` (dict 'train'/'val'; default global `dataloaders`),
    `amp` (fp16 autocast on CUDA), `mixup_alpha` (> 0: mixup), `epoch_callback(info)` (returns True to stop early),
    `verbose=False` (one compact line per epoch).
    """

    loaders = dataloaders if loaders is None else loaders
    use_amp = bool(amp) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    if not isinstance(num_epochs, numbers.Integral) or num_epochs < 1:
        raise ValueError('num_epochs must be a positive integer.')
    if not isinstance(criterion, nn.CrossEntropyLoss) or criterion.reduction != 'mean':
        raise ValueError('Use CrossEntropyLoss with reduction="mean".')
    if any(len(loaders[phase].dataset) == 0 for phase in ('train', 'val')):
        raise ValueError('Training and validation datasets must be nonempty.')

    seed_everything()

    if loaders['train'].generator is not None:
        loaders['train'].generator.manual_seed(SEED)

    criterion = criterion.to(device)

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = -float('inf')
    best_predictions = best_labels = None
    labels_order = np.arange(NUM_CLASSES)

    for epoch in range(num_epochs):
        if verbose:
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)

        for phase in ('train', 'val'):
            model.train(phase == 'train')
            running_loss = 0.0
            loss_denominator = 0.0
            phase_predictions, phase_labels = [], []

            for sample in loaders[phase]:
                inputs = sample['image'].to(device, dtype=torch.float32)
                labels = sample['label'].to(device, dtype=torch.long)

                # Mixing (applied to a fraction mix_prob of the training batches):
                #  - CutMix (cutmix_alpha > 0): paste a random box of a partner image; label weight = box area
                #  - mixup  (mixup_alpha > 0):  blend the two images
                #  cutmix_mode='risk' only pairs images of the 5 risk classes (Very Low..Very High) with each other;
                #  Water / Non-burnable images keep themselves as partner (unchanged).
                mixing = (phase == 'train' and (mixup_alpha > 0 or cutmix_alpha > 0)
                          and np.random.rand() < mix_prob)
                if mixing:
                    if cutmix_mode == 'risk':
                        perm = torch.arange(inputs.shape[0], device=inputs.device)
                        idx = torch.nonzero((labels >= 1) & (labels <= 5)).flatten()
                        perm[idx] = idx[torch.randperm(idx.numel(), device=inputs.device)]
                    else:
                        perm = torch.randperm(inputs.shape[0], device=inputs.device)
                    if cutmix_alpha > 0:
                        lam = float(np.random.beta(cutmix_alpha, cutmix_alpha))
                        H, W = inputs.shape[2], inputs.shape[3]
                        cut = math.sqrt(1.0 - lam)
                        ch, cw = int(H * cut), int(W * cut)
                        cy, cx = np.random.randint(H), np.random.randint(W)
                        y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H)
                        x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
                        inputs = inputs.clone()
                        inputs[:, :, y1:y2, x1:x2] = inputs[perm, :, y1:y2, x1:x2]
                        lam = 1.0 - (y2 - y1) * (x2 - x1) / float(H * W)   # real pasted area (box may hit the border)
                    else:
                        lam = float(np.random.beta(mixup_alpha, mixup_alpha))
                        inputs = lam * inputs + (1 - lam) * inputs[perm]

                if phase == 'train':
                    optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == 'train'):
                    with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                        logits = model(inputs)
                        if logits.shape != (labels.numel(), NUM_CLASSES):
                            raise ValueError('The network must return N x 7 logits.')
                        if mixing:
                            loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[perm])
                        else:
                            loss = criterion(logits, labels)
                    if not torch.isfinite(loss):
                        raise ValueError('Non-finite loss; check inputs and optimiser settings.')
                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                denominator = (labels.numel() if criterion.weight is None else
                               criterion.weight[labels].sum().item())
                running_loss += loss.item() * denominator
                loss_denominator += denominator
                phase_predictions.append(logits.detach().argmax(dim=1).cpu().numpy())
                phase_labels.append(labels.detach().cpu().numpy())   # original labels, also with mixup

            if phase == 'train' and scheduler is not None:
                scheduler.step()
            predictions = np.concatenate(phase_predictions)
            true_labels = np.concatenate(phase_labels)
            epoch_loss = running_loss / loss_denominator
            epoch_f1 = f1_score(true_labels, predictions, labels=labels_order,
                                average='macro', zero_division=0)
            if verbose:
                print(f'{phase} Loss: {epoch_loss:.4f} Macro F1-score: {epoch_f1:.4f}')
            if phase == 'train':
                train_loss, train_f1 = epoch_loss, epoch_f1
            if phase == 'val' and epoch_f1 > best_f1:
                best_f1 = epoch_f1
                best_model_wts = copy.deepcopy(model.state_dict())
                best_predictions = predictions.copy()
                best_labels = true_labels.copy()
        if verbose:
            print()
        else:
            print(f'ep {epoch + 1:2d}/{num_epochs}  train loss {train_loss:.3f} F1 {train_f1:.3f} | '
                  f'val loss {epoch_loss:.3f} F1 {epoch_f1:.3f}  (best {best_f1:.3f})')

        if epoch_callback is not None and epoch_callback(dict(
                epoch=epoch, train_loss=train_loss, train_f1=train_f1,
                val_loss=epoch_loss, val_f1=epoch_f1, best_f1=best_f1)):
            print(f'Pruned after epoch {epoch + 1}')
            break

    model.load_state_dict(best_model_wts)
    model.eval()

    elapsed = time.time() - since
    print(f'Training complete in {int(elapsed // 60)}m {int(elapsed % 60)}s')
    print(f'Best validation macro F1-score: {best_f1:.6f}')

    if plot_confusion_matrix:
        cm = confusion_matrix(best_labels, best_predictions, labels=labels_order,
                              normalize='true')
        display = ConfusionMatrixDisplay(100 * cm, display_labels=CLASS_NAMES)
        display.plot(xticks_rotation='vertical', values_format='.1f')
        plt.title('Row-normalised confusion matrix (%)')
        plt.tight_layout()
        plt.show()

    return model

In [15]:
image_datasets = {'train': train_dataset, 'val': val_dataset}

# dataloaders = {'train': train_dataloader, 'val': val_dataloader}

gpu_augment = GPUAugment(PIXEL_MEAN, PIXEL_STD)
dataloaders = {'train': GPULoader(train_dataloader, gpu_augment), 'val': val_dataloader}

dataset_sizes = {phase: len(dataset) for phase, dataset in image_datasets.items()}
class_names = CLASS_NAMES.copy()

In [16]:
SEARCH_DIR = WORK / 'search'          # small CSVs, versioned with git -> shared with the partner
MACHINE_ID = 1                       # <-- ONLY line that differs between the two computers (0 or 1)
N_MACHINES = 2                       # 1 = run every trial on this computer
SEARCH_EPOCHS = 15                   # budget per trial in phases 1-2
PRUNE_EPOCHS = (4, 8)                # median-stopping checks after these epochs
MIN_TRIALS_FOR_PRUNING = 4           # need this many finished trials of the phase before pruning
SEARCH_AMP = device.type == 'cuda'   # mixed precision: ~2x faster on the GPU
N_PHASE1, N_PHASE2 = 24, 16          # number of random trials in phase 1 / phase 2 (in total, over both computers)
N_FINALISTS, FINAL_EPOCHS = 3, 40    # phase 3: how many configs are retrained, and for how many epochs

# Fixed starting point: every search trial changes some of these values (the rest stay as they are here)
BASE_CFG = dict(base_channels=32, n_blocks=4,        # filters of block i = min(base_channels * 2**i, 512); number of blocks
                convs_per_block=1, head='gap',       # convs per block (VGG style); 'gap' = global avg pooling, 'flatten'
                hidden=128, dropout=0.3,             # fc1 units; dropout before the last layer
                optimizer='adamw', lr=1e-3, weight_decay=0.05, batch_size=64,
                warmup_epochs=1,                     # linear lr warm-up, then cosine decay
                label_smoothing=0.0, class_weights='none',   # class_weights: 'none' | 'sqrt_inv'
                mixup_alpha=0.0, aug='light')        # mixup strength (0 = off); GPU augmentation preset

SEARCH_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK / 'outputs' / 'Lab_4_2026_2027'
(OUTPUT_DIR / 'search').mkdir(parents=True, exist_ok=True)
print('SEARCH_DIR =', SEARCH_DIR, '| OUTPUT_DIR =', OUTPUT_DIR)

SEARCH_DIR = /kaggle/working/search | OUTPUT_DIR = /kaggle/working/outputs/Lab_4_2026_2027


In [20]:
# =====================================================================================================
# FINE-TUNING: settings, models, search space
# =====================================================================================================
import copy, gc, hashlib, json, math, random, statistics
from torchvision import models
from torchvision.ops import StochasticDepth

T0 = time.time()
MAX_HOURS = 11.0            # Kaggle kills a commit run at 12 h; nothing starts that would not finish before this
NUM_WORKERS = 4             # fixed: the crop RNG streams depend on it
PERSISTENT = NUM_WORKERS > 0
FT_DIR = OUTPUT_DIR / 'ft'
FT_DIR.mkdir(parents=True, exist_ok=True)

N_PHASE1 = 10               # random configurations per architecture (phase 1)
P1_EPOCHS = 10              # epochs per phase-1 trial
P1_PRUNE_AT = 4             # median stopping after this epoch (against finished trials of the SAME architecture)
P1_MIN_FOR_PRUNE = 3
N_FINALISTS = 2             # phase 2: best configurations of each architecture retrained longer
P2_EPOCHS = 30
FINAL_RESERVE_MIN = 10      # minutes kept for the final predictions and the submission

print('Deterministic algorithms:', torch.are_deterministic_algorithms_enabled(),
      '| cudnn.deterministic:', torch.backends.cudnn.deterministic, '| cudnn.benchmark:', torch.backends.cudnn.benchmark,
      '| CUBLAS_WORKSPACE_CONFIG:', os.environ.get('CUBLAS_WORKSPACE_CONFIG'), '| workers:', NUM_WORKERS,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

# ---- the three pretrained networks (+ ResNet-50 as reserve if one fails the compatibility/determinism test) ----
# lr = range of the BACKBONE learning rate (log-uniform); the new 7-class head gets lr * head_mult.
# sd = default stochastic-depth probability of the torchvision model (rescaled to the sampled drop_path).
ARCH_SPECS = {
    'convnext_tiny':     dict(short='cnx', builder=models.convnext_tiny,
                              weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1, lr=(2e-5, 3e-4), sd=0.1),
    'efficientnet_v2_s': dict(short='eff', builder=models.efficientnet_v2_s,
                              weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1, lr=(5e-5, 8e-4), sd=0.2),
    'swin_t':            dict(short='swin', builder=models.swin_t,
                              weights=models.Swin_T_Weights.IMAGENET1K_V1, lr=(2e-5, 3e-4), sd=0.2),
    'resnet50':          dict(short='r50', builder=models.resnet50,
                              weights=models.ResNet50_Weights.IMAGENET1K_V2, lr=(5e-5, 8e-4), sd=None),
}
ARCHS = ['convnext_tiny', 'efficientnet_v2_s', 'swin_t']
RESERVE = ['resnet50']

class GlobalAvgPool(nn.Module):
    """Same output as nn.AdaptiveAvgPool2d(1), but with a deterministic backward on CUDA
    (adaptive_avg_pool2d_backward has no deterministic implementation -> error in strict mode)."""
    def forward(self, x):
        return x.mean(dim=(2, 3), keepdim=True)

def replace_adaptive_pools(module):
    """Swap every AdaptiveAvgPool2d(1) (final pooling and the squeeze-excitation blocks of EfficientNet)."""
    for name, child in module.named_children():
        if isinstance(child, nn.AdaptiveAvgPool2d) and child.output_size in (1, (1, 1)):
            setattr(module, name, GlobalAvgPool())
        else:
            replace_adaptive_pools(child)
    return module

def make_model(arch, drop_path):
    """ImageNet-pretrained torchvision model with a NEW 7-output layer. Returns (model, head)."""
    spec = ARCH_SPECS[arch]
    m = spec['builder'](weights=spec['weights'])
    if arch == 'convnext_tiny':
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES); head = m.classifier
    elif arch == 'efficientnet_v2_s':
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES); head = m.classifier
    elif arch == 'swin_t':
        m.head = nn.Linear(m.head.in_features, NUM_CLASSES); head = m.head
    elif arch == 'resnet50':
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES); head = m.fc
    else:
        raise ValueError(arch)
    if spec['sd']:   # keep the per-block linear schedule of torchvision, scaled to the requested maximum
        for sd in m.modules():
            if isinstance(sd, StochasticDepth):
                sd.p = sd.p * drop_path / spec['sd']
    return replace_adaptive_pools(m), head

NO_DECAY_KEYS = ('relative_position_bias_table', 'layer_scale')

def param_groups(model, head, lr, head_mult, wd):
    """Backbone vs new head (head lr = lr * head_mult); no weight decay on biases, norms and layer-scale."""
    head_ids = {id(p) for p in head.parameters()}
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        part = 'head' if id(p) in head_ids else 'backbone'
        decay = p.ndim > 1 and not any(k in n for k in NO_DECAY_KEYS)
        groups.setdefault((part, decay), []).append(p)
    return [{'params': ps, 'lr': lr * (head_mult if part == 'head' else 1), 'weight_decay': wd if decay else 0.0}
            for (part, decay), ps in groups.items()]

def build(cfg, epochs):
    """Model, loss, optimiser, scheduler and loaders for one configuration (seed 42 everywhere)."""
    seed_everything(SEED)
    model, head = make_model(cfg['arch'], cfg['drop_path'])
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(cfg['class_weights']),
                                    label_smoothing=cfg['label_smoothing'])
    optimizer = optim.AdamW(param_groups(model, head, cfg['lr'], cfg['head_mult'], cfg['weight_decay']), lr=cfg['lr'])
    w = cfg['warmup_epochs']   # stepped once per epoch: linear warm-up, then cosine decay to 0
    scheduler = lr_scheduler.LambdaLR(optimizer, lambda e: (e + 1) / (w + 1) if e < w else
                                      0.5 * (1 + math.cos(math.pi * (e - w) / max(1, epochs - w))))
    train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(SEED),
                              pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    loaders = {'train': GPULoader(train_loader, GPUAugment(PIXEL_MEAN, PIXEL_STD, **AUG_PRESETS[cfg['aug']])),
               'val': val_loader}
    return model, criterion, optimizer, scheduler, loaders

def default_cfg(arch):
    """A standard fine-tuning recipe: always trial 00 of each architecture."""
    lo, hi = ARCH_SPECS[arch]['lr']
    return dict(arch=arch, lr=_round3(math.sqrt(lo * hi)), head_mult=10, weight_decay=0.05, batch_size=32,
                warmup_epochs=1, label_smoothing=0.1, class_weights='sqrt_inv', mixup_alpha=0.0, aug='strong',
                drop_path=0.1)

def sample_configs(arch, n):
    """Deterministic random search (string seed -> same list on every machine and run)."""
    rng = random.Random(f'ft-{arch}-{SEED}')
    lo, hi = ARCH_SPECS[arch]['lr']
    short = ARCH_SPECS[arch]['short']
    configs = [{**default_cfg(arch), 'trial_id': f'{short}_p1_00'}]
    for i in range(1, n):
        configs.append(dict(arch=arch,
                            lr=_round3(_loguniform(rng, lo, hi)),
                            head_mult=rng.choice([1, 10]),
                            weight_decay=_round3(_loguniform(rng, 0.01, 0.1)),
                            batch_size=rng.choice([32, 64]),
                            warmup_epochs=rng.choice([1, 2]),
                            label_smoothing=rng.choice([0.0, 0.1]),
                            class_weights=rng.choice(['none', 'sqrt_inv']),
                            mixup_alpha=rng.choice([0.0, 0.0, 0.2]),
                            aug=rng.choice(['light', 'strong']),
                            drop_path=rng.choice([0.1, 0.2, 0.3]),
                            trial_id=f'{short}_p1_{i:02d}'))
    return configs

# ---- ImageNet weights: download now (needs Internet ON in the Kaggle notebook settings) ----
for arch in ARCHS + RESERVE:
    w = ARCH_SPECS[arch]['weights']
    try:
        w.get_state_dict(progress=False)
        print(f'{arch:18s} weights {w} ready')
    except Exception as e:
        raise RuntimeError(f'Could not download the ImageNet weights of {arch}. In Kaggle: Settings -> Internet ON. '
                           f'({e!r})')

for arch in ARCHS:
    print(f'\n{arch}: {len(sample_configs(arch, N_PHASE1))} phase-1 configurations')
    display(pd.DataFrame(sample_configs(arch, N_PHASE1)).set_index('trial_id'))

Deterministic algorithms: True | cudnn.deterministic: True | cudnn.benchmark: False | CUBLAS_WORKSPACE_CONFIG: :4096:8 | workers: 4 | GPU: Tesla T4
convnext_tiny      weights ConvNeXt_Tiny_Weights.IMAGENET1K_V1 ready
efficientnet_v2_s  weights EfficientNet_V2_S_Weights.IMAGENET1K_V1 ready
swin_t             weights Swin_T_Weights.IMAGENET1K_V1 ready
resnet50           weights ResNet50_Weights.IMAGENET1K_V2 ready

convnext_tiny: 10 phase-1 configurations


,arch,lr,head_mult,weight_decay,batch_size,warmup_epochs,label_smoothing,class_weights,mixup_alpha,aug,drop_path
trial_id,,,,,,,,,,,
cnx_p1_00,convnext_tiny,0.000077,10,0.0500,32,1,0.1,sqrt_inv,0.0,strong,0.1
cnx_p1_01,convnext_tiny,0.000063,1,0.0230,64,1,0.1,none,0.0,light,0.3
cnx_p1_02,convnext_tiny,0.000034,10,0.0479,64,2,0.0,none,0.0,strong,0.2
cnx_p1_03,convnext_tiny,0.000123,10,0.0853,64,1,0.1,sqrt_inv,0.2,strong,0.3
cnx_p1_04,convnext_tiny,0.000039,10,0.0473,32,2,0.0,none,0.0,light,0.2
cnx_p1_05,convnext_tiny,0.000206,1,0.0642,64,1,0.0,sqrt_inv,0.0,light,0.1
cnx_p1_06,convnext_tiny,0.000187,1,0.0188,32,2,0.0,none,0.2,light,0.1
cnx_p1_07,convnext_tiny,0.000065,10,0.0265,64,1,0.1,none,0.2,light,0.1
cnx_p1_08,convnext_tiny,0.000260,10,0.0161,32,2,0.0,sqrt_inv,0.0,strong,0.3



efficientnet_v2_s: 10 phase-1 configurations


,arch,lr,head_mult,weight_decay,batch_size,warmup_epochs,label_smoothing,class_weights,mixup_alpha,aug,drop_path
trial_id,,,,,,,,,,,
eff_p1_00,efficientnet_v2_s,0.000200,10,0.0500,32,1,0.1,sqrt_inv,0.0,strong,0.1
eff_p1_01,efficientnet_v2_s,0.000140,1,0.0272,64,1,0.0,none,0.0,light,0.2
eff_p1_02,efficientnet_v2_s,0.000682,1,0.0853,32,2,0.1,sqrt_inv,0.0,light,0.3
eff_p1_03,efficientnet_v2_s,0.000435,1,0.0944,32,1,0.1,sqrt_inv,0.0,light,0.3
eff_p1_04,efficientnet_v2_s,0.000055,1,0.0723,64,2,0.0,none,0.0,strong,0.2
eff_p1_05,efficientnet_v2_s,0.000069,1,0.0826,32,1,0.0,sqrt_inv,0.2,light,0.1
eff_p1_06,efficientnet_v2_s,0.000372,10,0.0140,64,1,0.1,sqrt_inv,0.2,strong,0.3
eff_p1_07,efficientnet_v2_s,0.000442,10,0.0186,64,2,0.1,none,0.0,light,0.1
eff_p1_08,efficientnet_v2_s,0.000071,1,0.0163,32,2,0.0,none,0.2,strong,0.2



swin_t: 10 phase-1 configurations


,arch,lr,head_mult,weight_decay,batch_size,warmup_epochs,label_smoothing,class_weights,mixup_alpha,aug,drop_path
trial_id,,,,,,,,,,,
swin_p1_00,swin_t,0.000077,10,0.0500,32,1,0.1,sqrt_inv,0.0,strong,0.1
swin_p1_01,swin_t,0.000024,1,0.0395,32,2,0.0,sqrt_inv,0.0,light,0.2
swin_p1_02,swin_t,0.000235,10,0.0223,32,1,0.0,sqrt_inv,0.0,strong,0.1
swin_p1_03,swin_t,0.000131,10,0.0909,32,1,0.0,none,0.0,strong,0.1
swin_p1_04,swin_t,0.000026,1,0.0570,32,1,0.1,sqrt_inv,0.0,light,0.1
swin_p1_05,swin_t,0.000271,10,0.0722,32,1,0.1,none,0.2,light,0.1
swin_p1_06,swin_t,0.000122,10,0.0494,64,1,0.1,sqrt_inv,0.2,light,0.1
swin_p1_07,swin_t,0.000075,1,0.0477,32,2,0.1,sqrt_inv,0.2,light,0.1
swin_p1_08,swin_t,0.000022,1,0.0698,32,1,0.1,sqrt_inv,0.0,light,0.1


In [19]:
import json, math, statistics

# GPU augmentation presets ('light' == the GPUAugment defaults)
AUG_PRESETS = {'light': dict(gamma=0.1, brightness=0.05, contrast=0.05, saturation=0.05),
               'strong': dict(gamma=0.2, brightness=0.15, contrast=0.15, saturation=0.15)}

def class_weight_tensor(kind):
    """None, or inverse-sqrt-frequency class weights (mean 1) computed on the training labels."""
    if kind == 'none':
        return None
    counts = np.bincount(train_dataset.dataset['label'], minlength=NUM_CLASSES)[:NUM_CLASSES]
    w = 1 / np.sqrt(np.maximum(counts, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

def build_run(cfg, epochs):
    """Model, loss, optimiser, scheduler and loaders for one configuration."""
    seed_everything()
    channels = tuple(min(cfg['base_channels'] * 2 ** i, 512) for i in range(cfg['n_blocks']))
    model = CustomNet(channels=channels, kernel_size=3, padding=1, hidden=cfg['hidden'], dropout=cfg['dropout'],
                      batch_norm=True, convs_per_block=cfg['convs_per_block'], head=cfg['head'],
                      input_size=INPUT_SIZE).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(cfg['class_weights']),
                                    label_smoothing=cfg['label_smoothing'])
    if cfg['optimizer'] == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    else:
        optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, nesterov=True,
                              weight_decay=cfg['weight_decay'])
    w = cfg['warmup_epochs']   # stepped once per epoch: linear warm-up, then cosine decay to 0
    scheduler = lr_scheduler.LambdaLR(optimizer, lambda e: (e + 1) / (w + 1) if e < w else
                                      0.5 * (1 + math.cos(math.pi * (e - w) / max(1, epochs - w))))
    train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(SEED),
                              pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    loaders = {'train': GPULoader(train_loader, GPUAugment(PIXEL_MEAN, PIXEL_STD, **AUG_PRESETS[cfg['aug']])),
               'val': val_dataloader}
    return model, criterion, optimizer, scheduler, loaders

# ---- search spaces (deterministic: same list on both computers) ----
def _loguniform(rng, lo, hi):
    return math.exp(rng.uniform(math.log(lo), math.log(hi)))

def _round3(v):
    return float(f'{v:.3g}')   # 3 significant digits, keeps the CSV readable

def sample_phase1(n, seed=SEED):
    """Primary hyperparameters: optimiser, lr, width, depth, convs per block, head."""
    rng, configs = random.Random(seed + 1), []
    for i in range(n):
        cfg = dict(BASE_CFG)
        cfg['optimizer'] = rng.choice(['adamw', 'sgd'])
        if cfg['optimizer'] == 'adamw':
            cfg['lr'], cfg['weight_decay'] = _round3(_loguniform(rng, 2e-4, 3e-3)), 0.05
        else:
            cfg['lr'], cfg['weight_decay'] = _round3(_loguniform(rng, 5e-3, 1e-1)), 5e-4
        cfg['base_channels'] = rng.choice([24, 32, 48])
        cfg['n_blocks'] = rng.choice([4, 5])
        cfg['convs_per_block'] = rng.choice([1, 2])
        cfg['head'] = rng.choice(['gap', 'flatten'])
        cfg['trial_id'] = f'p1_{i:02d}'
        configs.append(cfg)
    return configs

def sample_phase2(best, n, seed=SEED):
    """Secondary hyperparameters around `best` (keeps its architecture and optimiser)."""
    rng, configs = random.Random(seed + 2), []
    for i in range(n):
        cfg = dict(best)
        cfg['lr'] = _round3(best['lr'] * 2 ** rng.uniform(-1, 1))
        lo, hi = (5e-3, 0.2) if best['optimizer'] == 'adamw' else (1e-4, 2e-3)
        cfg['weight_decay'] = _round3(_loguniform(rng, lo, hi))
        cfg['dropout'] = rng.choice([0.0, 0.2, 0.3, 0.5])
        cfg['hidden'] = rng.choice([128, 256])
        cfg['batch_size'] = rng.choice([32, 64, 128])
        cfg['label_smoothing'] = rng.choice([0.0, 0.1])
        cfg['class_weights'] = rng.choice(['none', 'sqrt_inv'])
        cfg['mixup_alpha'] = rng.choice([0.0, 0.2, 0.4])
        cfg['aug'] = rng.choice(['light', 'strong'])
        cfg['trial_id'], cfg['parent'] = f'p2_{i:02d}', best['trial_id']
        configs.append(cfg)
    return configs

# ---- results store: one CSV per machine and phase in SEARCH_DIR (merged when reading) ----
RESULT_COLS = ['trial_id', 'phase', 'machine', 'status', 'best_val_f1', 'best_epoch', 'epochs_run',
               'minutes', 'params', 'gpu', 'val_f1_history', 'cfg']

def read_results(phase=None):
    """All trials of every machine (optionally one phase), best validation macro F1 first."""
    files = sorted(SEARCH_DIR.glob('results_*.csv'))
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=RESULT_COLS)
    if phase is not None:
        df = df[df['phase'] == phase]
    return df.sort_values('best_val_f1', ascending=False).reset_index(drop=True)

def show_leaderboard(phase=None, top=10):
    df = read_results(phase).head(top)
    if df.empty:
        print('No results yet.')
        return
    cfgs = pd.DataFrame([json.loads(c) for c in df['cfg']]).drop(columns=['trial_id'], errors='ignore')
    out = pd.concat([df.drop(columns=['val_f1_history', 'cfg']).reset_index(drop=True), cfgs], axis=1)
    out['best_val_f1'] = out['best_val_f1'].round(4)
    display(out)

def best_config(phase=None, rank=0):
    """cfg (with trial_id) of the given rank of the merged leaderboard (done or pruned trials)."""
    df = read_results(phase)
    row = df[df['status'].isin(['done', 'pruned'])].iloc[rank]
    return {**json.loads(row['cfg']), 'trial_id': row['trial_id']}

def append_result(row, phase):
    """Append one row to this machine's CSV right after the trial (resume-safe)."""
    path = SEARCH_DIR / f'results_{phase}_m{MACHINE_ID}.csv'
    pd.DataFrame([row], columns=RESULT_COLS).to_csv(path, mode='a', header=not path.exists(), index=False)

def _row(cfg, phase, status, history, params=0, minutes=0.0):
    return dict(trial_id=cfg['trial_id'], phase=phase, machine=MACHINE_ID, status=status,
                best_val_f1=max(history, default=0.0), best_epoch=int(np.argmax(history)) + 1 if history else 0,
                epochs_run=len(history), minutes=round(minutes, 2), params=params,
                gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
                val_f1_history=json.dumps([round(float(h), 4) for h in history]), cfg=json.dumps(cfg))

def run_trial(cfg, phase, epochs, prune=True):
    """Train one config. Median stopping: after epochs in PRUNE_EPOCHS stop if the trial's best val F1 so far
    is below the median of what the finished trials of the same phase had reached by the same epoch."""
    since = time.time()
    model, criterion, optimizer, scheduler, loaders = build_run(cfg, epochs)
    history = []
    finished = [json.loads(h) for h in read_results(phase)['val_f1_history']]

    def cb(info):
        history.append(info['val_f1'])
        e = len(history)
        if not prune or e not in PRUNE_EPOCHS:
            return False
        reference = [max(h[:e]) for h in finished if len(h) >= e]
        if len(reference) >= MIN_TRIALS_FOR_PRUNING and max(history) < statistics.median(reference):
            print(f'Pruning: best val F1 {max(history):.3f} after {e} epochs < median {statistics.median(reference):.3f} '
                  f'of {len(reference)} finished trials')
            return True
        return False

    model = train_model(model, criterion, optimizer, scheduler, num_epochs=epochs, plot_confusion_matrix=False,
                        loaders=loaders, amp=SEARCH_AMP, mixup_alpha=cfg['mixup_alpha'], epoch_callback=cb, verbose=False)
    status = 'pruned' if len(history) < epochs else 'done'
    return model, _row(cfg, phase, status, history, sum(p.numel() for p in model.parameters()),
                       (time.time() - since) / 60)

def run_phase(phase, configs, epochs=SEARCH_EPOCHS, prune=True):
    """Run this machine's share of `configs` (index % N_MACHINES == MACHINE_ID), skipping trials already stored."""
    import gc
    mine = [c for i, c in enumerate(configs) if i % N_MACHINES == MACHINE_ID]
    done_ids = set(read_results()['trial_id'])
    for cfg in mine:
        if cfg['trial_id'] in done_ids:
            continue
        print(f"\n=== {cfg['trial_id']} (machine {MACHINE_ID}) {cfg}")
        try:
            model, row = run_trial(cfg, phase, epochs, prune)
        except torch.cuda.OutOfMemoryError:
            print('CUDA out of memory -> trial skipped')
            model, row = None, _row(cfg, phase, 'oom', [])
        append_result(row, phase)
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    show_leaderboard(phase)

In [23]:
## PARA GENERAR OUTPUT_FT.CSV CON OTROS MODELOS .pt

# ---- Predicciones FT con otros finalistas -> ft_extra/output_ft_2.csv (+ ensemble) ----
FT_DIR_IN = Path('/kaggle/input/datasets/csarlvarezcascos/fine-tuning-outputs/FineTuning_Output/outputs/Lab_4_2026_2027/ft')   # <- pega aquí tu ruta
FT_PT = {t: FT_DIR_IN / f'{t}.pt' for t in ['r50_p2_0', 'swin_p2_1', 'swin_p2_0']}
for t, p in FT_PT.items():
    print(t, p, 'OK' if p.exists() else 'NO EXISTE')

print('Checkpoints encontrados:', sorted(FT_PT))

def predict_probs(model, dataset):
    model.eval(); out = []
    with torch.inference_mode():
        for s in DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0):
            out.append(torch.softmax(model(s['image'].to(device, dtype=torch.float32)).float(), dim=1).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

VAL_LABELS = val_dataset.dataset['label'].to_numpy()
TRIALS = ['swin_p2_0', 'r50_p2_0', 'swin_p2_1']     # 1º modelo de la lista genera el de output_ft_2.csv; la media de las predicciones de todos 
OUT = WORK / 'ft_extra'; OUT.mkdir(parents=True, exist_ok=True)

test_p, val_p = {}, {}
for tid in TRIALS:
    ck = torch.load(FT_PT[tid], map_location=device, weights_only=False)
    m, _ = make_model(ck['cfg']['arch'], ck['cfg']['drop_path'])
    m.load_state_dict(ck['state_dict']); m = m.to(device).eval()
    val_p[tid], test_p[tid] = predict_probs(m, val_dataset), predict_probs(m, test_dataset)
    print(f"{tid}: val F1 {f1_score(VAL_LABELS, val_p[tid].argmax(1), average='macro'):.4f}", flush=True)
    del m; torch.cuda.empty_cache()

def save(name, p):
    p = p / p.sum(axis=1, keepdims=True)
    assert p.shape == (len(test_dataset), NUM_CLASSES) and np.isfinite(p).all()
    np.savetxt(OUT / name, p, delimiter=',', fmt='%.9g'); print('Guardado', OUT / name, p.shape)

save('output_ft_3.csv', test_p[TRIALS[0]])
ens_val = np.mean([val_p[t] for t in TRIALS], axis=0)
print(f"Ensemble val F1: {f1_score(VAL_LABELS, ens_val.argmax(1), average='macro'):.4f}")
save('output_ft_ensemble.csv', np.mean([test_p[t] for t in TRIALS], axis=0))

r50_p2_0 /kaggle/input/datasets/csarlvarezcascos/fine-tuning-outputs/FineTuning_Output/outputs/Lab_4_2026_2027/ft/r50_p2_0.pt OK
swin_p2_1 /kaggle/input/datasets/csarlvarezcascos/fine-tuning-outputs/FineTuning_Output/outputs/Lab_4_2026_2027/ft/swin_p2_1.pt OK
swin_p2_0 /kaggle/input/datasets/csarlvarezcascos/fine-tuning-outputs/FineTuning_Output/outputs/Lab_4_2026_2027/ft/swin_p2_0.pt OK
Checkpoints encontrados: ['r50_p2_0', 'swin_p2_0', 'swin_p2_1']
swin_p2_0: val F1 0.5323
r50_p2_0: val F1 0.5407
swin_p2_1: val F1 0.5457
Guardado /kaggle/working/ft_extra/output_ft_3.csv (1173, 7)
Ensemble val F1: 0.5446
Guardado /kaggle/working/ft_extra/output_ft_ensemble.csv (1173, 7)


In [ ]:
# =====================================================================================================
# Compatibility + determinism test: each network, default recipe, 1 epoch twice -> identical weights?
# Also measures the seconds per epoch used by the time planner. A network that fails is replaced by ResNet-50.
# =====================================================================================================
def weights_of(m):
    return {k: v.detach().float().cpu().clone() for k, v in m.state_dict().items()}

DET_FILE = FT_DIR / 'determinism_check.json'
det = json.loads(DET_FILE.read_text()) if DET_FILE.exists() else {}
reserve = list(RESERVE)
checked = []
queue = list(ARCHS)
while queue:
    arch = queue.pop(0)
    if arch not in det:
        try:
            states, secs = [], []
            for _ in range(2):
                cfg = default_cfg(arch)
                m, crit, opt, sch, ld = build(cfg, 1)
                t = time.time()
                m = train_model(m, crit, opt, sch, num_epochs=1, plot_confusion_matrix=False, loaders=ld,
                                amp=SEARCH_AMP, mixup_alpha=0.0, verbose=False)
                secs.append(time.time() - t)
                states.append(weights_of(m)); del m, opt; gc.collect(); torch.cuda.empty_cache()
            diff = max((states[0][k] - states[1][k]).abs().max().item() for k in states[0])
            det[arch] = {'max_abs_weight_diff': diff, 'identical': diff == 0.0, 'sec_per_epoch': min(secs),
                         'params': sum(v.numel() for v in states[0].values())}
            del states
        except Exception as e:
            det[arch] = {'error': repr(e)}
            gc.collect(); torch.cuda.empty_cache()
        DET_FILE.write_text(json.dumps({**det, 'torch': torch.__version__, 'torchvision': __import__('torchvision').__version__,
                                        'cuda': torch.version.cuda}, indent=1))
    print(arch, det[arch])
    if 'error' in det[arch] or not det[arch].get('identical', False):
        if reserve:
            sub = reserve.pop(0)
            print(f'  -> {arch} is not usable/deterministic here: replaced by {sub}')
            queue.insert(0, sub)
        continue
    checked.append(arch)

ARCHS = checked
SEC_PER_EPOCH = {a: det[a]['sec_per_epoch'] for a in ARCHS}
print('\nArchitectures in the search:', ARCHS)
p1 = sum(N_PHASE1 * P1_EPOCHS * SEC_PER_EPOCH[a] for a in ARCHS) / 60
p2 = sum(N_FINALISTS * P2_EPOCHS * SEC_PER_EPOCH[a] for a in ARCHS) / 60
print(f'Estimated: phase 1 <= {p1:.0f} min (less with pruning), phase 2 {p2:.0f} min, '
      f'total <= {(p1 + p2) / 60:.1f} h of the {MAX_HOURS} h budget')

In [ ]:
# =====================================================================================================
# Search: phase 1 (random configs, short, median pruning) -> phase 2 (finalists, long) -> leaderboard
# =====================================================================================================
RESULTS_CSV = FT_DIR / 'ft_results.csv'
PERCLASS_CSV = FT_DIR / 'ft_per_class_f1.csv'
RES_COLS = ['trial_id', 'phase', 'arch', 'status', 'best_val_f1', 'best_epoch', 'last5_val_f1', 'epochs_run',
            'train_f1_at_best', 'minutes', 'val_f1_history', 'cfg']

def read_results():
    return pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=RES_COLS)

def predict_probs(model, dataset):
    """Softmax probabilities in dataset order, fp32 (no autocast), no TTA."""
    model.eval(); out = []
    with torch.inference_mode():
        for s in DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2):
            out.append(torch.softmax(model(s['image'].to(device, dtype=torch.float32)).float(), dim=1).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

VAL_LABELS = val_dataset.dataset['label'].to_numpy()

def minutes_left():
    return MAX_HOURS * 60 - (time.time() - T0) / 60

def est_minutes(arch, epochs):
    return 1.15 * epochs * SEC_PER_EPOCH[arch] / 60 + 1

def run_trial(cfg, phase, epochs, prune):
    since = time.time()
    model, crit, opt, sch, loaders = build(cfg, epochs)
    prev = read_results()
    finished = [json.loads(h) for h in prev[(prev['phase'] == phase) & (prev['arch'] == cfg['arch']) &
                                            (prev['status'].isin(['done', 'pruned']))]['val_f1_history']]
    log = []
    def cb(info):
        log.append({'epoch': info['epoch'] + 1, 'train_loss': info['train_loss'], 'train_f1': info['train_f1'],
                    'val_loss': info['val_loss'], 'val_f1': info['val_f1'], 'lr': opt.param_groups[0]['lr']})
        e = len(log)
        if prune and e == P1_PRUNE_AT:
            ref = [max(h[:e]) for h in finished if len(h) >= e]
            best = max(r['val_f1'] for r in log)
            if len(ref) >= P1_MIN_FOR_PRUNE and best < statistics.median(ref):
                print(f'Pruning: best val F1 {best:.3f} after {e} epochs < median {statistics.median(ref):.3f}')
                return True
        return False
    model = train_model(model, crit, opt, sch, num_epochs=epochs, plot_confusion_matrix=False, loaders=loaders,
                        amp=SEARCH_AMP, mixup_alpha=cfg['mixup_alpha'], epoch_callback=cb, verbose=False)
    d = pd.DataFrame(log)
    d.to_csv(FT_DIR / f"{cfg['trial_id']}_epochs.csv", index=False)
    hist = d['val_f1'].tolist()
    b = int(np.argmax(hist))
    row = dict(trial_id=cfg['trial_id'], phase=phase, arch=cfg['arch'],
               status='pruned' if len(hist) < epochs else 'done', best_val_f1=round(hist[b], 4), best_epoch=b + 1,
               last5_val_f1=round(float(np.mean(hist[-5:])), 4), epochs_run=len(hist),
               train_f1_at_best=round(float(d['train_f1'][b]), 4), minutes=round((time.time() - since) / 60, 2),
               val_f1_history=json.dumps([round(float(h), 4) for h in hist]), cfg=json.dumps(cfg))
    pd.DataFrame([row], columns=RES_COLS).to_csv(RESULTS_CSV, mode='a', header=not RESULTS_CSV.exists(), index=False)
    SEC_PER_EPOCH[cfg['arch']] = max(SEC_PER_EPOCH[cfg['arch']], 60 * row['minutes'] / max(1, len(hist)))
    return model, row

def leaderboard(phase=None):
    r = read_results()
    if phase:
        r = r[r['phase'] == phase]
    if r.empty:
        return r
    cfgs = pd.DataFrame([json.loads(c) for c in r['cfg']]).drop(columns=['trial_id', 'arch'], errors='ignore')
    out = pd.concat([r.drop(columns=['val_f1_history', 'cfg']).reset_index(drop=True), cfgs], axis=1)
    return out.sort_values('best_val_f1', ascending=False).reset_index(drop=True)

def phase2_need(ranks_left):
    return sum(est_minutes(a, P2_EPOCHS) * ranks_left for a in ARCHS)

# ---------------- PHASE 1: round robin over the architectures (all get coverage if time runs short) -------------
P1 = {a: sample_configs(a, N_PHASE1) for a in ARCHS}
done_ids = set(read_results()['trial_id'])
for i in range(N_PHASE1):
    for arch in ARCHS:
        cfg = P1[arch][i]
        if cfg['trial_id'] in done_ids:
            continue
        need = est_minutes(arch, P1_EPOCHS)
        if need + phase2_need(N_FINALISTS) + FINAL_RESERVE_MIN > minutes_left():
            print(f"{cfg['trial_id']}: skipped (time is reserved for phase 2)"); continue
        print(f"\n##### PHASE 1 {cfg['trial_id']} | {(time.time() - T0) / 60:.0f} min elapsed | {cfg}")
        try:
            model, row = run_trial(cfg, 'p1', P1_EPOCHS, prune=True)
            print(f"=== {row['trial_id']}: best {row['best_val_f1']:.4f} (ep {row['best_epoch']}) {row['status']}")
            del model
        except torch.cuda.OutOfMemoryError:
            print('CUDA out of memory -> trial skipped')
        except Exception as e:
            print(f"{cfg['trial_id']}: FAILED with {e!r}")
        gc.collect(); torch.cuda.empty_cache()

lb1 = leaderboard('p1')
lb1.round(4).to_csv(FT_DIR / 'ft_leaderboard_phase1.csv', index=False)
display(lb1.round(4))

# ---------------- PHASE 2: best N_FINALISTS configs of each architecture, P2_EPOCHS epochs, checkpoint saved ----
def per_class_row(name, model):
    pred = predict_probs(model, val_dataset).argmax(1)
    pc = f1_score(VAL_LABELS, pred, labels=np.arange(NUM_CLASSES), average=None, zero_division=0)
    pd.DataFrame([[name, *pc, pc.mean()]], columns=['trial_id', *CLASS_NAMES, 'macro_fp32']).to_csv(
        PERCLASS_CSV, mode='a', header=not PERCLASS_CSV.exists(), index=False)
    return pc.mean()

finalists = []
for rank in range(N_FINALISTS):
    for arch in ARCHS:
        sub = lb1[(lb1['arch'] == arch) & (lb1['status'].isin(['done', 'pruned']))]
        if len(sub) > rank:
            parent = sub.iloc[rank]['trial_id']
            cfg = {**json.loads(read_results().set_index('trial_id').loc[parent, 'cfg'])}
            cfg.update(trial_id=f"{ARCH_SPECS[arch]['short']}_p2_{rank}", parent=parent)
            finalists.append((rank, cfg))

done_ids = set(read_results()['trial_id'])
for k, (rank, cfg) in enumerate(finalists):
    if cfg['trial_id'] in done_ids and (FT_DIR / f"{cfg['trial_id']}.pt").exists():
        continue
    if est_minutes(cfg['arch'], P2_EPOCHS) + FINAL_RESERVE_MIN > minutes_left():
        print(f"{cfg['trial_id']}: skipped, not enough time left"); continue
    print(f"\n##### PHASE 2 {cfg['trial_id']} (from {cfg['parent']}) | {(time.time() - T0) / 60:.0f} min elapsed")
    try:
        model, row = run_trial(cfg, 'p2', P2_EPOCHS, prune=False)
        torch.save({'state_dict': model.state_dict(), 'cfg': cfg, 'best_val_f1_amp': row['best_val_f1'],
                    'best_epoch': row['best_epoch']}, FT_DIR / f"{cfg['trial_id']}.pt")
        f32 = per_class_row(cfg['trial_id'], model)
        print(f"=== {row['trial_id']}: best {row['best_val_f1']:.4f} (ep {row['best_epoch']}), last5 "
              f"{row['last5_val_f1']:.4f}, fp32 reload {f32:.4f}, {row['minutes']:.0f} min")
        del model
    except torch.cuda.OutOfMemoryError:
        print('CUDA out of memory -> run skipped')
    except Exception as e:
        print(f"{cfg['trial_id']}: FAILED with {e!r}")
    gc.collect(); torch.cuda.empty_cache()

lb2 = leaderboard('p2')
lb2.round(4).to_csv(FT_DIR / 'ft_leaderboard_phase2.csv', index=False)
display(lb2.round(4))
if PERCLASS_CSV.exists():
    display(pd.read_csv(PERCLASS_CSV).round(3))
print(f'Search finished after {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# =====================================================================================================
# Final FT model = best phase-2 run by validation macro-F1 -> output_ft.csv (+ CodaBench zip if output_custom.csv
# is among the inputs). Probabilities in fp32, no TTA, test images in CSV order.
# =====================================================================================================
cands = [p for p in FT_DIR.glob('*_p2_*.pt')]
if not cands:
    raise RuntimeError('No phase-2 checkpoint was produced.')
lb2 = leaderboard('p2')
lb2 = lb2[lb2['trial_id'].isin([p.stem for p in cands])]
best = lb2.iloc[0]
ckpt_path = FT_DIR / f"{best['trial_id']}.pt"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['cfg']
model, _ = make_model(cfg['arch'], cfg['drop_path'])
model.load_state_dict(ckpt['state_dict'])
model = model.to(device).eval()

val_probs = predict_probs(model, val_dataset)
val_f1 = f1_score(VAL_LABELS, val_probs.argmax(1), labels=np.arange(NUM_CLASSES), average='macro', zero_division=0)
print(f"Final FT model: {best['trial_id']} ({cfg['arch']}), val macro-F1 {val_f1:.4f} fp32 "
      f"(training log with AMP: {best['best_val_f1']:.4f} at epoch {best['best_epoch']})")

probs = predict_probs(model, test_dataset)
probs = probs / probs.sum(axis=1, keepdims=True)
assert probs.shape == (len(test_dataset), NUM_CLASSES) and np.isfinite(probs).all()
assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-6, rtol=0) and (probs >= 0).all()
FINAL_DIR = WORK / 'final_ft'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
np.savetxt(FINAL_DIR / 'output_ft.csv', probs, delimiter=',', fmt='%.9g')
counts = np.bincount(probs.argmax(1), minlength=NUM_CLASSES)
print('Test predicted counts:', dict(zip(CLASS_NAMES, counts.tolist())))

custom = sorted(Path('/kaggle/input').rglob('output_custom.csv')) if IN_KAGGLE else []
if custom:
    shutil.copy(custom[0], FINAL_DIR / 'output_custom.csv')
    c = np.loadtxt(FINAL_DIR / 'output_custom.csv', delimiter=',')
    assert c.shape == probs.shape, f'output_custom.csv has shape {c.shape}'
    with zipfile.ZipFile(FINAL_DIR / 'codabench_submission.zip', 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for f in ('output_custom.csv', 'output_ft.csv'):
            z.write(FINAL_DIR / f, arcname=f)
    print('codabench_submission.zip written with', custom[0])
else:
    print('No output_custom.csv among the inputs: only output_ft.csv was written (zip it with the custom one later).')

manifest = {'model': 'FINE-TUNING - ImageNet-pretrained torchvision ' + cfg['arch'],
            'weights': str(ARCH_SPECS[cfg['arch']]['weights']), 'cfg': cfg, 'trial_id': best['trial_id'],
            'val_macro_f1_fp32': round(float(val_f1), 6), 'best_epoch': int(best['best_epoch']),
            'epochs': P2_EPOCHS, 'seed': SEED, 'input': '224 crop (random train / centre val+test), ImageNet mean/std',
            'checkpoint_sha256': hashlib.sha256(ckpt_path.read_bytes()).hexdigest(),
            'output_ft_sha256': hashlib.sha256((FINAL_DIR / 'output_ft.csv').read_bytes()).hexdigest(),
            'torch': torch.__version__, 'torchvision': __import__('torchvision').__version__,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
            'test_pred_counts': counts.tolist()}
(FINAL_DIR / 'ft_final_manifest.json').write_text(json.dumps(manifest, indent=1))
shutil.copy(ckpt_path, FINAL_DIR / f"ft_final_{best['trial_id']}.pt")
print(json.dumps(manifest, indent=1))
for p in sorted(list(FT_DIR.iterdir()) + list(FINAL_DIR.iterdir())):
    print(f'  {str(p.relative_to(WORK)):55s} {p.stat().st_size / 1e6:8.2f} MB')